In [1]:
!pip install transformers datasets scikit-learn sentencepiece accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 223.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 59.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaPreTrainedModel, RobertaModel, Trainer, TrainerCallback
from google.colab import drive
import gc

print("Loading datasets...")
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/NLPLabs-2024/Dont_Patronize_Me_Trainingset/'

#load data
df_raw = pd.read_csv(base_path + 'dontpatronizeme_pcl.tsv', sep='\t', skiprows=4, header=None, on_bad_lines='skip')
df_raw.columns = ['par_id', 'art_id', 'keyword', 'country', 'text', 'label']
df_raw['par_id'] = df_raw['par_id'].astype(str)

# Clean labels
df_raw = df_raw.dropna(subset=['text', 'label'])
df_raw['label'] = pd.to_numeric(df_raw['label'], errors='coerce')
df_raw = df_raw.dropna(subset=['label'])

df_raw['binary_label'] = df_raw['label'].apply(lambda x: 1 if x >= 2 else 0).astype(int)

# load splits
train_ids = pd.read_csv(base_path + 'train_semeval_parids-labels.csv')
dev_ids = pd.read_csv(base_path + 'dev_semeval_parids-labels.csv')
train_ids['par_id'] = train_ids['par_id'].astype(str)
dev_ids['par_id'] = dev_ids['par_id'].astype(str)

# merge
df_train = pd.merge(df_raw, train_ids[['par_id']], on='par_id', how='inner')
df_dev = pd.merge(df_raw, dev_ids[['par_id']], on='par_id', how='inner')

# test set
df_test = pd.read_csv(base_path + 'task4_test.tsv', sep='\t', header=None, on_bad_lines='skip')
df_test.columns = ['par_id', 'art_id', 'keyword', 'country', 'text']
df_test['text'] = df_test['text'].fillna("") # Prevent NaN errors

# df_train['text'] = df_train['keyword'] + " | " + df_train['country'].astype(str) + " : " + df_train['text']
# df_dev['text'] = df_dev['keyword'] + " | " + df_dev['country'].astype(str) + " : " + df_dev['text']
# df_test['text'] = df_test['keyword'] + " | " + df_test['country'].astype(str) + " : " + df_test['text']

print(f"Training samples: {len(df_train)}")
print(f"Dev samples: {len(df_dev)}")
print(f"Test samples: {len(df_test)}")

Loading datasets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training samples: 8375
Dev samples: 2093
Test samples: 3832


In [6]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings['input_ids'])

In [7]:
!pip install textaugment transformers[sentencepiece] sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 123.4 MB/s eta 0:00:00


In [8]:
from transformers import MarianMTModel, MarianTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Models (En -> Fr and Fr -> En)
en_fr_name = "Helsinki-NLP/opus-mt-en-fr"
fr_en_name = "Helsinki-NLP/opus-mt-fr-en"

en_fr_tokenizer = MarianTokenizer.from_pretrained(en_fr_name)
en_fr_model = MarianMTModel.from_pretrained(en_fr_name).to(device)

fr_en_tokenizer = MarianTokenizer.from_pretrained(fr_en_name)
fr_en_model = MarianMTModel.from_pretrained(fr_en_name).to(device)

def back_translate_batch(texts):
    # En -> Fr
    inputs = en_fr_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    translated = en_fr_model.generate(**inputs)
    fr_texts = [en_fr_tokenizer.decode(t, skip_special_tokens=True) for t in translated]

    # Fr -> En
    inputs_back = fr_en_tokenizer(fr_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    back_translated = fr_en_model.generate(**inputs_back)
    return [fr_en_tokenizer.decode(t, skip_special_tokens=True) for t in back_translated]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [9]:
import pandas as pd
from tqdm import tqdm

# Filter only PCL cases from the training set (label 1)
pcl_df = df_train[df_train['binary_label'] == 1].copy()
texts_to_aug = pcl_df['text'].tolist()

print(f"Augmenting {len(texts_to_aug)} PCL samples...")
augmented_texts = []
batch_size = 16

for i in tqdm(range(0, len(texts_to_aug), batch_size)):
    batch = texts_to_aug[i:i+batch_size]
    augmented_texts.extend(back_translate_batch(batch))

df_aug = pcl_df.copy()
df_aug['text'] = augmented_texts

df_train_boosted = pd.concat([df_train, df_aug], ignore_index=True).sample(frac=1, random_state=42)

print(f"New training size: {len(df_train_boosted)} (Original: {len(df_train)})")

Augmenting 794 PCL samples...


New training size: 9169 (Original: 8375)


In [10]:
from textaugment import EDA
import pandas as pd

t = EDA()

# Augmenting the PCL samples (binary_label == 1)
pcl_df = df_train_boosted[df_train_boosted['binary_label'] == 1].copy()
aug_texts = []

print("Performing EDA...")
for text in pcl_df['text']:
    # Generate one new variation for every PCL sentence
    aug_texts.append(t.synonym_replacement(text))

df_eda_aug = pd.DataFrame({'text': aug_texts, 'binary_label': [1]*len(aug_texts)})
df_final_train = pd.concat([df_train_boosted, df_eda_aug], ignore_index=True).sample(frac=1).reset_index(drop=True)
print(f"Final training size: {len(df_final_train)}")

Performing EDA...
Final training size: 10757


In [11]:
import torch
import torch.nn as nn
from transformers import RobertaPreTrainedModel, RobertaModel

class RobertaForPCL(RobertaPreTrainedModel):
    _keys_to_ignore_on_load_missing = [r"position_ids"]

    def __init__(self, config, weights=None):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = RobertaModel(config, add_pooling_layer=False)

        # Multi-sample dropout: 5 masks to improve stability
        self.dropouts = nn.ModuleList([nn.Dropout(0.3) for _ in range(5)])
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.weights = weights
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)

        sequence_output = outputs[0][:, 0, :]

        logits = torch.mean(torch.stack([self.classifier(d(sequence_output)) for d in self.dropouts], dim=0), dim=0)

        loss = None
        if labels is not None:
            # Label Smoothing (0.1) handles subjective/noisy labels
            loss_fct = nn.CrossEntropyLoss(
                weight=self.weights.to(sequence_output.device) if self.weights is not None else None,
                label_smoothing=0.1
            )
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return (loss, logits) if loss is not None else logits

In [12]:
from transformers import TrainerCallback

class LayerFreezingCallback(TrainerCallback):
    def __init__(self, unfreeze_epoch=2):
        self.unfreeze_epoch = unfreeze_epoch

    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        # Freeze layers in epochs 0 and 1; unfreeze starting at epoch 2
        if state.epoch < self.unfreeze_epoch:
            print(f"Epoch {state.epoch}: Freezing RoBERTa base layers...")
            for param in model.roberta.parameters():
                param.requires_grad = False
        else:
            print(f"Epoch {state.epoch}: Unfreezing all layers...")
            for param in model.roberta.parameters():
                param.requires_grad = True

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, DebertaV2ForSequenceClassification, DebertaV2Tokenizer

# Implement Focal Loss for severe class imbalances
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha # Class weights

    def forward(self, inputs, targets):
        # Dynamically cast the weights
        if self.alpha is not None:
            current_alpha = self.alpha.to(device=inputs.device, dtype=inputs.dtype)
        else:
            current_alpha = None

        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=current_alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Subclass HuggingFace Trainer to use our Focal Loss
class FocalTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = FocalLoss(alpha=self.class_weights, gamma=2.0)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [14]:
from sklearn.model_selection import StratifiedKFold
from transformers import TrainingArguments, RobertaTokenizer, get_cosine_schedule_with_warmup

roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_dev_ds = SimpleDataset(roberta_tokenizer(df_dev['text'].tolist(), truncation=True, padding=True, max_length=128))
roberta_test_ds = SimpleDataset(roberta_tokenizer(df_test['text'].tolist(), truncation=True, padding=True, max_length=128))

class_counts = df_final_train['binary_label'].value_counts().sort_index().values
weights_tensor = torch.tensor(len(df_final_train) / (2.0 * class_counts), dtype=torch.float)

# Arrays for RoBERTa
roberta_oof = np.zeros(len(df_final_train))
roberta_dev = np.zeros(len(df_dev))
roberta_test = np.zeros(len(df_test))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(df_final_train['text'], df_final_train['binary_label'])):
    print(f"\n========== ROBERTA FOLD {fold + 1}/5 ==========")
    train_ds = SimpleDataset(roberta_tokenizer(df_final_train['text'].iloc[train_idx].tolist(), truncation=True, padding=True, max_length=128), df_final_train['binary_label'].iloc[train_idx].tolist())
    val_ds = SimpleDataset(roberta_tokenizer(df_final_train['text'].iloc[val_idx].tolist(), truncation=True, padding=True, max_length=128), df_final_train['binary_label'].iloc[val_idx].tolist())

    model = RobertaForPCL.from_pretrained('roberta-base', weights=weights_tensor)

    optimizer = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if "roberta" in n], "lr": 1e-5},
        {"params": [p for n, p in model.named_parameters() if "classifier" in n], "lr": 5e-5}
    ], weight_decay=0.01)

    steps = (len(train_ds) // 32) * 5
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(steps * 0.1), steps)

    args = TrainingArguments(output_dir=f'./roberta_fold_{fold}', num_train_epochs=5, per_device_train_batch_size=16, gradient_accumulation_steps=2, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="eval_loss", fp16=True, save_total_limit=1, report_to="none")

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, optimizers=(optimizer, scheduler), callbacks=[LayerFreezingCallback()])
    trainer.train()

    roberta_oof[val_idx] = torch.nn.functional.softmax(torch.tensor(trainer.predict(val_ds).predictions), dim=-1)[:, 1].numpy()
    roberta_dev += torch.nn.functional.softmax(torch.tensor(trainer.predict(roberta_dev_ds).predictions), dim=-1)[:, 1].numpy() / 5
    roberta_test += torch.nn.functional.softmax(torch.tensor(trainer.predict(roberta_test_ds).predictions), dim=-1)[:, 1].numpy() / 5

    del model, trainer, train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()



========== ROBERTA FOLD 1/5 ==========


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Epoch 0: Freezing RoBERTa base layers...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3.0: Unfreezing all layers...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4.0: Unfreezing all layers...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, get_cosine_schedule_with_warmup, TrainerCallback
import torch
import torch.nn as nn
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold

print("=== STARTING EXPLOSION-PROOF DEBERTA ===")

MODEL_NAME = 'microsoft/deberta-v3-base'
N_SPLITS = 5
MAX_LEN = 128
EPOCHS = 4
LR = 2e-5

deberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

deberta_dev_ds = SimpleDataset(deberta_tokenizer(df_dev['text'].tolist(), truncation=True, padding=True, max_length=MAX_LEN))
deberta_test_ds = SimpleDataset(deberta_tokenizer(df_test['text'].tolist(), truncation=True, padding=True, max_length=MAX_LEN))

# DeBERTa Freezing Callback
class DebertaFreezingCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        # Freeze the backbone for the first epoch to let the classification head settle
        if state.epoch < 1.0:
            print(f"Epoch {state.epoch}: Freezing DeBERTa backbone...")
            for param in model.deberta.parameters():
                param.requires_grad = False
        else:
            print(f"Epoch {state.epoch}: Unfreezing all layers...")
            for param in model.deberta.parameters():
                param.requires_grad = True

# Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class_counts = df_final_train['binary_label'].value_counts().sort_index().values
weights_tensor = torch.tensor(len(df_final_train) / (2.0 * class_counts), dtype=torch.float)

deberta_oof_probs = np.zeros(len(df_final_train))
deberta_dev_probs = np.zeros(len(df_dev))
deberta_test_probs = np.zeros(len(df_test))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(df_final_train['text'], df_final_train['binary_label'])):
    print(f"\n========== DEBERTA FOLD {fold + 1}/{N_SPLITS} ==========")

    train_ds = SimpleDataset(deberta_tokenizer(df_final_train['text'].iloc[train_idx].tolist(), truncation=True, padding=True, max_length=MAX_LEN), df_final_train['binary_label'].iloc[train_idx].tolist())
    val_ds = SimpleDataset(deberta_tokenizer(df_final_train['text'].iloc[val_idx].tolist(), truncation=True, padding=True, max_length=MAX_LEN), df_final_train['binary_label'].iloc[val_idx].tolist())

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, eps=1e-6) # eps=1e-6 IS THE FIX!
    num_training_steps = (len(train_ds) // (16 * 2)) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(num_training_steps * 0.1), num_training_steps)

    training_args = TrainingArguments(
        output_dir=f'./deberta_results_fold_{fold}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=16, # Increased to 16 since we are using 'base'
        gradient_accumulation_steps=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=True,
        adam_epsilon=1e-6, # Hugging Face argument for the optimizer fix
        max_grad_norm=1.0,
        report_to="none",
        save_total_limit=1
    )

    trainer = WeightedTrainer(
        class_weights=weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        optimizers=(optimizer, scheduler),
        callbacks=[DebertaFreezingCallback()] # Adding our freezing callback
    )

    trainer.train()

    print("Predicting Validation...")
    deberta_oof_probs[val_idx] = torch.nn.functional.softmax(torch.tensor(trainer.predict(val_ds).predictions), dim=-1)[:, 1].numpy()

    print("Predicting Official Dev and Test...")
    deberta_dev_probs += torch.nn.functional.softmax(torch.tensor(trainer.predict(deberta_dev_ds).predictions), dim=-1)[:, 1].numpy() / N_SPLITS
    deberta_test_probs += torch.nn.functional.softmax(torch.tensor(trainer.predict(deberta_test_ds).predictions), dim=-1)[:, 1].numpy() / N_SPLITS

    del model, trainer, train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

print("\nDeBERTa Training Complete!")

=== STARTING EXPLOSION-PROOF DEBERTA ===


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


========== DEBERTA FOLD 1/5 ==========


...

Epoch 3.0: Unfreezing all layers...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Predicting Validation...


Predicting Official Dev and Test...



DeBERTa Training Complete!


In [19]:
from sklearn.metrics import precision_recall_curve, f1_score

print("=== CREATING THE RO-DEBERTA ENSEMBLE ===")
# Soft Voting (Averaging)
blended_oof = (roberta_oof + deberta_oof) / 2.0
blended_dev = (roberta_dev + deberta_dev) / 2.0
blended_test = (roberta_test + deberta_test) / 2.0

# Find the optimal threshold for the blend
labels_all = df_final_train['binary_label'].values
precisions, recalls, thresholds = precision_recall_curve(labels_all, blended_oof)
f1_scores = np.divide(2 * (precisions * recalls), (precisions + recalls), out=np.zeros_like(precisions), where=(precisions + recalls) != 0)

opt_thresh = thresholds[np.argmax(f1_scores)]
print(f"Optimal Ensemble Threshold: {opt_thresh:.4f}")

final_dev_preds = (blended_dev >= opt_thresh).astype(int)
final_test_preds = (blended_test >= opt_thresh).astype(int)

dev_f1 = f1_score(df_dev['binary_label'].values, final_dev_preds)
print(f">>> FINAL SUPER-MODEL DEV F1-SCORE: {dev_f1:.4f} <<<")

# Save Final Outputs
with open("dev.txt", "w") as f:
    for p in final_dev_preds: f.write(f"{p}\n")
with open("test.txt", "w") as f:
    for p in final_test_preds: f.write(f"{p}\n")
print("\nSuccess! 'dev.txt' and 'test.txt' generated.")

=== CREATING THE RO-DEBERTA ENSEMBLE ===
Optimal Ensemble Threshold: 0.3090
>>> FINAL SUPER-MODEL DEV F1-SCORE: 0.5714 <<<

Success! 'dev.txt' and 'test.txt' generated.


In [27]:
import os
import glob
import json
import shutil

drive_base_path = '/content/drive/My Drive/NLPLabs-2024/'
deberta_save_path = os.path.join(drive_base_path, 'Best_DeBERTa_Model')

# Copy the text files to your Drive
for file_name in ["dev.txt", "test.txt"]:
    if os.path.exists(file_name):
        shutil.copy(file_name, os.path.join(drive_base_path, file_name))
        print(f"Saved {file_name} to Google Drive!")
    else:
        print(f"Could not find {file_name}")

print("\n=== 2. FINDING & SAVING THE BEST DEBERTA MODEL ===")
best_loss = float('inf')
best_dir = ""

state_files = glob.glob('./deberta_results_fold_*/**/trainer_state.json', recursive=True)

if not state_files:
    print("No DeBERTa log files found!")
else:
    for file_path in state_files:
        try:
            with open(file_path, 'r') as f:
                state = json.load(f)
                loss = float(state.get('best_metric', 999.0))
                if loss < best_loss:
                    best_loss = loss
                    best_dir = os.path.dirname(file_path)
        except Exception:
            pass

    print(f"Found the Best DeBERTa Fold at: {best_dir} (Loss: {best_loss:.4f})")
    print("Copying to Google Drive...")

    # Remove old folder if it exists to prevent mixing weights
    if os.path.exists(deberta_save_path):
        shutil.rmtree(deberta_save_path)

    shutil.copytree(best_dir, deberta_save_path)
    print("Successfully saved Best DeBERTa to Google Drive!")


Saved dev.txt to Google Drive!
Saved test.txt to Google Drive!

=== 2. FINDING & SAVING THE BEST DEBERTA MODEL ===
Found the Best DeBERTa Fold at: ./deberta_results_fold_2/checkpoint-269 (Loss: 0.6538)
Copying to Google Drive...
Successfully saved Best DeBERTa to Google Drive!


In [26]:
import os
import glob
import json
import shutil
from google.colab import files

best_roberta_loss = float('inf')
best_roberta_dir = ""

# Search the RoBERTa folders we just created
roberta_state_files = glob.glob('./roberta_fold_*/**/trainer_state.json', recursive=True)

if not roberta_state_files:
    print("No RoBERTa log files found! Check the folder names.")
else:
    for file_path in roberta_state_files:
        try:
            with open(file_path, 'r') as f:
                state = json.load(f)
                loss = float(state.get('best_metric', 999.0))
                if loss < best_roberta_loss:
                    best_roberta_loss = loss
                    best_roberta_dir = os.path.dirname(file_path)
        except Exception:
            pass

    print(f"Found the Best RoBERTa Fold at: {best_roberta_dir} (Loss: {best_roberta_loss:.4f})")

    drive_base_path = '/content/drive/My Drive/NLPLabs-2024/'
    roberta_save_path = os.path.join(drive_base_path, 'Best_RoBERTa_Model')

    print("Copying to Google Drive...")

    if os.path.exists(roberta_save_path):
        shutil.rmtree(roberta_save_path)

    shutil.copytree(best_roberta_dir, roberta_save_path)
    print("Successfully saved Best RoBERTa to Google Drive!")


Found the Best RoBERTa Fold at: ./roberta_fold_2/checkpoint-1345 (Loss: 0.3926)
Copying to Google Drive...
Successfully saved Best RoBERTa to Google Drive!


In [24]:
import pandas as pd

results_df = df_dev.copy()
results_df['prediction'] = final_dev_preds

false_positives = results_df[(results_df['prediction'] == 1) & (results_df['binary_label'] == 0)]
false_negatives = results_df[(results_df['prediction'] == 0) & (results_df['binary_label'] == 1)]

print("=== 3 FALSE POSITIVES (Model predicted PCL, but it wasn't) ===")
for text in false_positives['text'].head(3):
    print(f"- {text}\n")

print("=== 3 FALSE NEGATIVES (Model missed actual PCL) ===")
for text in false_negatives['text'].head(3):
    print(f"- {text}\n")

=== 3 FALSE POSITIVES (Model predicted PCL, but it wasn't) ===
- Marcos said the government should help poor families that try every possible means to survive . With Joel Zurbano <h> More from this Category :

- As she descended into a state of hopelessness , she decided to do " one hopeful thing " . She bought a Golden Retriever puppy called Bunker . Little did she know how much solace he would provide ...

- Accessibility and the rights of disabled New Zealanders , people with access needs , older people and anyone who is currently struggling to access the world around them , is the social movement of our time .

=== 3 FALSE NEGATIVES (Model missed actual PCL) ===
- His present " chambers " may be quite humble , but Shiyani has the tiny space very neatly organized and clean . Many people pass him by but do not manage to see him , because the space is partially hidden behind trees , which gives him a relative privacy . " There are many homeless sleeping around the station , " Captain 

In [25]:
from sklearn.metrics import confusion_matrix, f1_score

# 1. Confusion Matrix
tn, fp, fn, tp = confusion_matrix(results_df['binary_label'], results_df['prediction']).ravel()
print("=== CONFUSION MATRIX NUMBERS ===")
print(f"True Negatives (TN): {tn}")
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}\n")

# 2. Category-Specific Breakdown
print("=== F1-SCORE BY KEYWORD ===")
keywords = results_df['keyword'].unique()
best_kw, worst_kw = "", ""
best_f1, worst_f1 = -1.0, 2.0

for kw in keywords:
    subset = results_df[results_df['keyword'] == kw]
    # Only calculate if there are actual positive PCL examples in this subset
    if sum(subset['binary_label']) > 0:
        score = f1_score(subset['binary_label'], subset['prediction'])
        print(f"Keyword '{kw}': F1 = {score:.4f}")

        if score > best_f1:
            best_f1 = score
            best_kw = kw
        if score < worst_f1:
            worst_f1 = score
            worst_kw = kw

print(f"\nBEST Keyword: '{best_kw}' ({best_f1:.4f})")
print(f"WORST Keyword: '{worst_kw}' ({worst_f1:.4f})")

=== CONFUSION MATRIX NUMBERS ===
True Negatives (TN): 1778
True Positives (TP): 126
False Positives (FP): 116
False Negatives (FN): 73

=== F1-SCORE BY KEYWORD ===
Keyword 'homeless': F1 = 0.5797
Keyword 'disabled': F1 = 0.4138
Keyword 'poor-families': F1 = 0.5789
Keyword 'in-need': F1 = 0.7674
Keyword 'women': F1 = 0.5600
Keyword 'refugee': F1 = 0.4000
Keyword 'migrant': F1 = 0.3636
Keyword 'hopeless': F1 = 0.4516
Keyword 'immigrant': F1 = 0.4444
Keyword 'vulnerable': F1 = 0.6364

BEST Keyword: 'in-need' (0.7674)
WORST Keyword: 'migrant' (0.3636)
